# Phase 2 — Data Acquisition & Inspection

Run this notebook in **Google Colab** (Runtime → Change runtime type → GPU is *not* needed for this notebook, it's just data prep — save your GPU quota for Phases 3 and 5).

This notebook downloads and inspects the four datasets for the grocery VQA assistant:
1. **VizWiz-VQA** — real photos + questions from blind users (core VQA fine-tuning set)
2. **Freiburg Groceries** — grocery product classification
3. **ExpDate** — expiration date recognition
4. **Grozi-120 / Grocery Products** — product recognition

Each section: download → unzip → inspect a few real examples, so you actually see what you're training on before we write any training code.

## Setup

In [ ]:
!pip install -q gdown
import os, json, zipfile, requests
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)

## 1. VizWiz-VQA

Source: https://vizwiz.org/tasks-and-datasets/vqa/

We download the **validation** split first (4,319 images — small, fast, good for inspection and eval).
Training images (20,523) download the same way — just swap `val` for `train` — but do that later, only once
you're ready to actually fine-tune (Phase 5), since it's a much bigger download.

In [ ]:
vizwiz_dir = DATA_DIR / "vizwiz"
vizwiz_dir.mkdir(exist_ok=True)

# Annotations (small, get these first)
!wget -q -O {vizwiz_dir}/annotations.zip https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/Annotations.zip
!unzip -q -o {vizwiz_dir}/annotations.zip -d {vizwiz_dir}

# Validation images (~4,319 images, manageable size for inspection)
!wget -q -O {vizwiz_dir}/val_images.zip https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip
!unzip -q -o {vizwiz_dir}/val_images.zip -d {vizwiz_dir}/val_images

print("Done. Contents:")
!ls {vizwiz_dir}

In [ ]:
# Inspect real examples
with open(vizwiz_dir / "val.json") as f:
    val_data = json.load(f)

print(f"Total validation Q&A pairs: {len(val_data)}\n")

for item in val_data[:5]:
    print("Image:   ", item["image"])
    print("Question:", item["question"])
    print("Answers: ", [a["answer"] for a in item["answers"]])
    print("---")

In [ ]:
# Visualize a few image + question + answer examples
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, item in zip(axes, val_data[:3]):
    img_path = vizwiz_dir / "val_images" / item["image"]
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(f"Q: {item['question']}\nA: {item['answers'][0]['answer']}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Freiburg Groceries Dataset

Source: https://github.com/PhilJd/freiburg_groceries_dataset
5,000 images across 25 grocery classes.

In [ ]:
freiburg_dir = DATA_DIR / "freiburg"
freiburg_dir.mkdir(exist_ok=True)

!git clone -q https://github.com/PhilJd/freiburg_groceries_dataset.git {freiburg_dir}/repo
!cd {freiburg_dir}/repo/src && python3 download_dataset.py

!ls {freiburg_dir}/repo

## 3. ExpDate — Expiration Date Dataset

Source: https://acseker.github.io/ExpDateWebsite/
Hosted on Google Drive (six sub-datasets: Products-Real, Products-Synth, Date-Real, Date-Synth, Components-Real, Components-Synth).
Start with **Products-Real** (1,767 real images) — that's the one directly relevant to "read the expiration date off this package.

In [ ]:
expdate_dir = DATA_DIR / "expdate"
expdate_dir.mkdir(exist_ok=True)

# Replace FOLDER_ID with the Products-Real subfolder ID from the Drive link on the ExpDate site
# (the full collection is at https://drive.google.com/drive/folders/1YuxWzVj6bT6gs6XlewEGetYrdgwZt7EH)
!gdown --folder https://drive.google.com/drive/folders/1YuxWzVj6bT6gs6XlewEGetYrdgwZt7EH -O {expdate_dir}

!ls {expdate_dir}

## 4. Grozi-120 / Grocery Products Dataset

Product recognition dataset (120 categories, web + in-store images). We'll confirm the current
working mirror/download link together before running this cell — the original hosting has moved
around over the years, so check for the active link before Phase 3.

In [ ]:
# grozi_dir = DATA_DIR / "grozi120"
# TODO: confirm current download URL before running
# !wget -q -O {grozi_dir}.zip <URL>


## Summary

Run this once all four datasets are downloaded, to confirm counts before moving to Phase 3.

In [ ]:
print("VizWiz val Q&A pairs:", len(val_data))
print("VizWiz val images:", len(list((vizwiz_dir / 'val_images').glob('*.jpg'))))
print("Freiburg classes:", len(list((freiburg_dir / 'repo' / 'images').iterdir())) if (freiburg_dir / 'repo' / 'images').exists() else 'check download_dataset.py output above')